# Experiment 2 — Stage-1 candidate training + selection (cascade-pid-72n)

Trains the three Stage-1 candidates as constrained-decoding **benign/injection
classifiers** (`p_safe`) via the canonical `scripts/train_stage1.py` (bead
a40.2), then selects the winner on the validation split by **DR@1%FPR**.

**Candidates:** `qwen2.5-1.5b`, `llama3.2-1b`, `granite-guardian-2b`
(`configs/models/*.yaml`).

**Per-candidate outputs** (E2 convention, `results/stage1/<name>/`):
`adapter/`, `val_logits.jsonl` (5,724 rows), `cal_logits.jsonl` (5,722 rows),
`train_summary.json`. Selection → `results/stage1/selection.json`.

**Selection metric:** DR@1%FPR overall (tie-break: per-channel recall on the
uncontaminated **direct**/**tool-output** channels, then latency).

**cloud-gpu handoff:** real training runs on Colab GPU (`RUN_MODE="full"`).
On MPS/CPU the notebook does a 1-step dry-run only. Agent verifies the artifacts
(row counts, `p_safe` range) locally before closing the bead.

**Payload hygiene (CLAUDE.md):** no dataset text is printed — cells report only
counts, `p_safe` ranges, metrics, and paths. Clear outputs before saving.

In [ ]:
# ── Environment detection + installs ────────────────────────────────────
# Auto-detects Google Colab and installs the QLoRA training stack there.
# On a local machine this cell is a no-op (uses the project environment).
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    %pip install -q -U bitsandbytes accelerate peft transformers trl datasets
    # Colab preinstalls torchao 0.10, below peft's minimum (>=0.16); peft raises
    # on a merely-present old torchao even though we never use it (quantization
    # is bitsandbytes). Removing is safer than upgrading torch's CUDA build.
    %pip uninstall -q -y torchao
    print("Colab detected — training dependencies installed.")
else:
    print(f"Local run ({sys.platform}) — using the project environment as-is.")


In [ ]:
# ── Config ─────────────────────────────────────────────────────────
# BASELINE_SPEC.md §Environment ; results/analysis/STAGE2_BASE_MODEL_DESIGN_NOTE.md

RUN_MODE = "smoke"   # "smoke" | "full"   <- set "full" for the GPU Colab run
SEED     = 3131

import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    # Project synced to Drive; scripts/ configs/ data/ live inside it.
    REPO_ROOT = Path("/content/drive/MyDrive/Thesis")
    # Gated bases (llama3.2, mistral) need an HF token — Colab Secrets (key icon).
    if "HF_TOKEN" not in os.environ:
        try:
            from google.colab import userdata
            os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or ""
        except Exception:
            pass
else:
    _here = Path(globals().get("__vsc_ipynb_file__", Path.cwd() / "_")).resolve()
    REPO_ROOT = next((p for p in _here.parents if (p / "BASELINE_SPEC.md").exists()), Path.cwd())

# make repo importable (src/ , scripts/)
for _p in (str(REPO_ROOT), str(REPO_ROOT / "src")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

CANDIDATES = ["qwen2.5-1.5b", "llama3.2-1b", "granite-guardian-2b"]
MODELS_DIR = REPO_ROOT / "configs" / "models"
TRAIN_FILE = REPO_ROOT / "data" / "train_proposal" / "train.jsonl"
VAL_FILE   = REPO_ROOT / "data" / "train_proposal" / "val.jsonl"
RESULTS_DIR = REPO_ROOT / "results" / "stage1"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

assert TRAIN_FILE.exists(), f"train.jsonl not found at {TRAIN_FILE}"
for name in CANDIDATES:
    cfg = MODELS_DIR / f"{name}.yaml"
    assert cfg.exists(), f"missing model config {cfg}"
print("candidates:", CANDIDATES)

print(f"IN_COLAB={IN_COLAB}  RUN_MODE={RUN_MODE}  SEED={SEED}")
print(f"REPO_ROOT={REPO_ROOT}")


## Device detection

QLoRA 4-bit needs a CUDA GPU (bitsandbytes is CUDA-only). On MPS/CPU the
notebook falls back to a 1-step **dry-run** that validates the pipeline without
real training — the real run is the `cloud-gpu` Colab pass.

In [ ]:
import torch

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

CUDA_AVAILABLE = (DEVICE == "cuda")
# Real training only on CUDA + RUN_MODE="full"; otherwise a 1-step dry-run.
DRY_RUN = not (CUDA_AVAILABLE and RUN_MODE == "full")
print(f"Detected device: {DEVICE}   CUDA={CUDA_AVAILABLE}   DRY_RUN={DRY_RUN}")

if DRY_RUN:
    print(
        "\n" + "=" * 70 + "\n"
        "Real QLoRA training needs a CUDA GPU + RUN_MODE='full'.\n"
        f"This machine reports device={DEVICE}, RUN_MODE={RUN_MODE}.\n"
        "\nProceeding in DRY-RUN mode: 1 optimizer step on a tiny subset to\n"
        "validate the pipeline (no logit dumps, no selection). For the real run:\n"
        "  1. Sync the project to Google Drive at MyDrive/Thesis.\n"
        "  2. Open in Colab with a GPU runtime, set RUN_MODE='full', Run all.\n"
        + "=" * 70
    )


In [ ]:
import subprocess

def run_train(config_path, out_dir, train_file=None, training_cfg=None):
    """Shell out to the canonical, tested trainer (scripts/train_stage1.py,
    bead a40.2). Returns the process return code; streams logs live."""
    cmd = [sys.executable, str(REPO_ROOT / "scripts" / "train_stage1.py"),
           "--config", str(config_path), "--output-dir", str(out_dir)]
    if train_file is not None:
        cmd += ["--train-file", str(train_file)]
    if training_cfg is not None:
        cmd += ["--training", str(training_cfg)]
    if DRY_RUN:
        cmd += ["--dry-run"]
    print("$ " + " ".join(cmd), flush=True)
    proc = subprocess.run(cmd, cwd=str(REPO_ROOT))
    return proc.returncode


In [ ]:
import json, math

def sanity_logits(out_dir, n_val=5724, n_cal=5722):
    """Verify val/cal logit dumps match the split row counts and p_safe range.
    Payload-safe: reads only {logp_benign, logp_injection, p_safe}, never text."""
    report = {}
    for stem, n_exp in (("val", n_val), ("cal", n_cal)):
        fp = Path(out_dir) / f"{stem}_logits.jsonl"
        if not fp.exists():
            report[stem] = {"exists": False}
            continue
        n = 0; bad = 0; pmin = 1.0; pmax = 0.0
        with fp.open() as fh:
            for line in fh:
                if not line.strip():
                    continue
                r = json.loads(line); n += 1
                p = r.get("p_safe")
                if p is None or not (0.0 <= p <= 1.0) or math.isnan(p) \
                   or "logp_benign" not in r or "logp_injection" not in r:
                    bad += 1
                else:
                    pmin = min(pmin, p); pmax = max(pmax, p)
        report[stem] = {"exists": True, "rows": n, "expected": n_exp,
                        "rows_ok": n == n_exp, "bad_rows": bad,
                        "p_safe_min": round(pmin, 4), "p_safe_max": round(pmax, 4)}
    return report


In [ ]:
from evaluation.metrics import detection_rate_at_fpr

_SAFE = {"safe", "benign", 0, "0"}

def _load_val_meta(val_path):
    """label(0/1) + channel per val row, IN ORDER. Never reads `input`."""
    labels, channels = [], []
    with open(val_path) as fh:
        for line in fh:
            if not line.strip():
                continue
            r = json.loads(line)
            labels.append(0 if r.get("label") in _SAFE else 1)
            channels.append(r.get("channel"))
    return labels, channels

def _load_scores(logits_path):
    """injection score = 1 - p_safe, IN ORDER."""
    scores = []
    with open(logits_path) as fh:
        for line in fh:
            if not line.strip():
                continue
            scores.append(1.0 - json.loads(line)["p_safe"])
    return scores

def eval_candidate(val_path, logits_path):
    labels, channels = _load_val_meta(val_path)
    scores = _load_scores(logits_path)
    assert len(labels) == len(scores), f"row mismatch: {len(labels)} vs {len(scores)}"
    dr = detection_rate_at_fpr(labels, scores, fpr_targets=(0.01,))["0.01"]
    thr = dr["threshold"]
    # per-channel injection recall at the global 1%-FPR threshold
    per_ch = {}
    from collections import defaultdict
    hit = defaultdict(int); tot = defaultdict(int)
    for lab, ch, sc in zip(labels, channels, scores):
        if lab == 1:
            tot[ch] += 1
            if sc > thr:
                hit[ch] += 1
    for ch in tot:
        per_ch[ch] = round(hit[ch] / tot[ch], 4) if tot[ch] else None
    return {"dr_at_1pct_fpr": round(dr["dr"], 4),
            "achieved_fpr": round(dr["achieved_fpr"], 4),
            "threshold": round(thr, 4), "per_channel_recall": per_ch}


## Train the candidates

In [ ]:
# ── Train each candidate ──────────────────────────────────────────
# Each writes results/stage1/<name>/ {adapter, val_logits.jsonl, cal_logits.jsonl,
# train_summary.json}. Logit dumps happen only on a full (non-dry) run.
train_rc = {}
for name in CANDIDATES:
    print("\n" + "#" * 70 + f"\n# {name}\n" + "#" * 70, flush=True)
    rc = run_train(MODELS_DIR / f"{name}.yaml", RESULTS_DIR / name, train_file=TRAIN_FILE)
    train_rc[name] = rc
    print(f"[{name}] return code = {rc}")

failed = [n for n, rc in train_rc.items() if rc != 0]
assert not failed, f"training failed for: {failed}"
print("\nall candidates trained OK:", train_rc)


## Sanity — logit dumps

In [ ]:
# ── Sanity: logit dumps match split row counts + p_safe range ─────────────
for name in CANDIDATES:
    rep = sanity_logits(RESULTS_DIR / name)
    print(f"{name}: {rep}")
    if not DRY_RUN:
        for stem in ("val", "cal"):
            assert rep[stem]["exists"] and rep[stem]["rows_ok"], f"{name}/{stem} row count wrong"
            assert rep[stem]["bad_rows"] == 0, f"{name}/{stem} has out-of-range/NaN p_safe"
if DRY_RUN:
    print("\n(DRY-RUN: logit dumps skipped — row-count asserts deferred to the full run.)")


## Selection — DR@1%FPR on val

In [ ]:
# ── Selection: DR@1%FPR on val ────────────────────────────────────
if DRY_RUN:
    print("DRY-RUN: no val_logits to select on. Run RUN_MODE='full' on GPU first.")
else:
    scores_by_model = {}
    for name in CANDIDATES:
        scores_by_model[name] = eval_candidate(VAL_FILE, RESULTS_DIR / name / "val_logits.jsonl")
        m = scores_by_model[name]
        print(f"{name:22s} DR@1%FPR={m['dr_at_1pct_fpr']:.4f} "
              f"(achieved_fpr={m['achieved_fpr']:.4f})  per-channel={m['per_channel_recall']}")

    # winner = max DR@1%FPR; tie-break mean recall on direct + tool-output
    def _key(name):
        m = scores_by_model[name]
        pc = m["per_channel_recall"]
        clean = [pc.get(c) for c in ("direct", "tool-output") if pc.get(c) is not None]
        return (m["dr_at_1pct_fpr"], sum(clean) / len(clean) if clean else 0.0)
    winner = max(CANDIDATES, key=_key)

    selection = {"task": "cascade-pid-72n", "metric": "DR@1%FPR (val)",
                 "seed": SEED, "candidates": scores_by_model, "winner": winner}
    out = RESULTS_DIR / "selection.json"
    out.write_text(json.dumps(selection, indent=2))
    print(f"\nWINNER: {winner}  → wrote {out}")


## Summary

In [ ]:
# ── Final summary ─────────────────────────────────────────────
print("=" * 60)
print("Stage-1 model selection — run complete")
print("=" * 60)
print(f"RUN_MODE : {RUN_MODE}    DRY_RUN : {DRY_RUN}    device : {DEVICE}")
for name in CANDIDATES:
    d = RESULTS_DIR / name
    print(f"  {name:22s} adapter={ (d/'adapter').exists() }  "
          f"summary={ (d/'train_summary.json').exists() }")
sel = RESULTS_DIR / "selection.json"
print(f"selection : {sel}  ({'written' if sel.exists() else 'pending full run'})")


## Colab instructions (full run)

1. **Sync to Drive.** Copy the project to `MyDrive/Thesis` so
   `scripts/`, `configs/`, `src/`, and `data/train_proposal/{train,val,cal}.jsonl`
   all live under it. Results land in `MyDrive/Thesis/results/stage1/` and
   survive disconnects.
2. **HF token.** Add `HF_TOKEN` in Colab Secrets (key icon, left) — Llama-3.2
   is gated. The Config cell reads it automatically.
3. **GPU runtime.** Runtime → Change runtime type → GPU (L4 / A100 recommended;
   T4 works but is slow). Set `RUN_MODE="full"` in the Config cell.
4. **Run all.** Trains all three candidates (~1.5B each; minutes on A100/L4),
   dumps val/cal logits, and writes `selection.json`.
5. **Verify before closing 72n.** Each `results/stage1/<name>/` has `adapter/`,
   `val_logits.jsonl` (5,724 rows), `cal_logits.jsonl` (5,722 rows),
   `train_summary.json`; `p_safe ∈ [0,1]`, no NaN; `selection.json` names a winner.